# Tutorial: Spin Operators via Fock-Space Occupation Representation

## Introduction

In many quantum-chemistry and condensed-matter codes, spin operators are built directly from **fermionic creation and annihilation operators** acting on an occupation-number (Fock-space) basis.  This approach is more flexible than the Pauli-matrix Kronecker-product construction used in the `Ising.ipynb` tutorial: it naturally handles

* different numbers of spin-up ($N_\alpha$) and spin-down ($N_\beta$) electrons,
* the single-occupancy (spin-½) constraint at each lattice site,
* fermionic anticommutation signs, and
* the construction of general **XYZ Heisenberg** Hamiltonians that go beyond the restricted XXZ model.

This tutorial introduces a new utility module, `moha.spin_utils`, which implements these ideas.  The presentation is transparent — using only NumPy — to show clearly how mathematical objects relate to concrete matrix representations.

---

## Outline
1. **Theory**: second-quantized spin operators from fermionic operators
2. **Fock-space basis**: enumeration, seniority, and $N_\alpha / N_\beta$ sectors
3. **Operator construction**: $S^+$, $S^-$, $S^z$ matrices from `spin_utils`
4. **Commutation-relation verification** (purely in NumPy)
5. **XYZ Heisenberg Hamiltonian**: build and diagonalise
6. **Benchmark**: compare with Pauli-matrix construction
7. **Special limits**: XXX (isotropic Bethe-ansatz), Ising ($J_x = J_y = 0$)
8. **Limitations and future work**


---
## 1  Theory

### 1.1  Fermionic creation / annihilation operators

Let site $k$ host two spin orbitals: $k\alpha$ (spin up) and $k\beta$ (spin down).  The creation operator $a^\dagger_{k\sigma}$ adds an electron to orbital $k\sigma$; the annihilation operator $a_{k\sigma}$ removes it.  They satisfy the **anticommutation relations**

$$\{a_{k\sigma},\, a^\dagger_{l\tau}\} = \delta_{kl}\,\delta_{\sigma\tau}, \qquad \{a_{k\sigma},\, a_{l\tau}\} = 0.$$

### 1.2  Spin-½ operators in second quantisation

The physical spin-raising, spin-lowering, and $z$-projection operators at site $k$ are

$$S_k^+ = a^\dagger_{k\alpha}\,a_{k\beta}, \qquad S_k^- = a^\dagger_{k\beta}\,a_{k\alpha}, \qquad S_k^z = \frac{1}{2}\!\left(n_{k\alpha} - n_{k\beta}\right),$$

where $n_{k\sigma} = a^\dagger_{k\sigma}a_{k\sigma}$ is the number operator.  They satisfy the standard $\mathfrak{su}(2)$ algebra:

$$[S^+, S^-] = 2S^z, \qquad [S^z, S^+] = +S^+, \qquad [S^z, S^-] = -S^-.$$

### 1.3  XYZ Heisenberg Hamiltonian

The most general spin-½ Heisenberg model with (possibly) different coupling strengths along each axis:

$$\hat{H}_{XYZ} = \sum_{p \neq q}\!\left[\frac{J^x_{pq}+J^y_{pq}}{2}\,S_p^+S_q^- + \frac{J^x_{pq}-J^y_{pq}}{4}\left(S_p^+S_q^+ + S_p^-S_q^-\right) + J^z_{pq}\,S_p^zS_q^z\right] + \sum_p \mu_p S_p^z.$$

> **Coupling convention.**  `build_xyz_hamiltonian` uses a *symmetric* coupling matrix $J_{pq} = J_{qp}$ and loops over **all ordered pairs** $(p,q)$ with $p\neq q$.  This means each physical bond is counted **twice** (once as $(p,q)$ and once as $(q,p)$).  To match the standard Pauli-matrix convention $H = J_{\text{phys}} \sum_{\langle pq\rangle}\mathbf{S}_p\cdot\mathbf{S}_q$ (where each bond appears once), set $J_{pq} = J_{\text{phys}}/2$ in the coupling matrix.

When $J^x = J^y$ the pairing terms vanish and we recover the **XXZ model** (also used in the `Ising.ipynb` tutorial).  Setting additionally $J^x = J^y = 0$ recovers the **Ising model**.


---
## 2  Fock-space basis

In the occupation-number representation, a many-body state for $L$ sites is encoded as two bit-strings

$$|\text{state}\rangle = |\underbrace{\alpha_0\,\alpha_1\,\cdots\,\alpha_{L-1}}_{\text{alpha bits}},\, \underbrace{\beta_0\,\beta_1\,\cdots\,\beta_{L-1}}_{\text{beta bits}}\rangle,$$

where $\alpha_k = 1$ (resp. $\beta_k = 1$) means site $k$ is occupied by an alpha (resp. beta) electron.  `spin_utils.fock_basis` enumerates all states with a fixed number $N_\alpha$ of alpha electrons and $N_\beta$ of beta electrons.

For spin-½ physics we require **single occupancy**: exactly one electron per site (either alpha *or* beta).  `spin_utils.spin_half_basis` provides this constrained basis, which has exactly $2^L$ states.

### 2.1  Seniority

In the spin-½ single-occupancy basis, the $(N_\alpha, N_\beta)$ sub-sector with $N_\alpha + N_\beta = L$ has a natural *seniority-like* quantum number $|N_\alpha - N_\beta|$ that measures the excess spin.


In [1]:
# ── imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '..')  # repo root: enables 'from moha.spin_utils import ...'

import numpy as np
from moha.spin_utils import (
    fock_basis, spin_half_basis,
    spin_plus_matrix, spin_minus_matrix, spin_z_matrix,
    verify_commutations, build_xyz_hamiltonian,
)

# ── 2-site system: enumerate all (N_alpha, N_beta) sectors ───────────────────
L = 2
print(f'L = {L} sites — all Fock sectors')
for na in range(L + 1):
    for nb in range(L + 1):
        basis = fock_basis(L, na, nb)
        print(f'  N_alpha={na}, N_beta={nb}: {len(basis):>2} states  {basis}')

L = 2 sites — all Fock sectors
  N_alpha=0, N_beta=0:  1 states  [(0, 0)]
  N_alpha=0, N_beta=1:  2 states  [(0, 1), (0, 2)]
  N_alpha=0, N_beta=2:  1 states  [(0, 3)]
  N_alpha=1, N_beta=0:  2 states  [(1, 0), (2, 0)]
  N_alpha=1, N_beta=1:  4 states  [(1, 1), (1, 2), (2, 1), (2, 2)]
  N_alpha=1, N_beta=2:  2 states  [(1, 3), (2, 3)]
  N_alpha=2, N_beta=0:  1 states  [(3, 0)]
  N_alpha=2, N_beta=1:  2 states  [(3, 1), (3, 2)]
  N_alpha=2, N_beta=2:  1 states  [(3, 3)]


In [2]:
# Physical spin-½ basis for L = 2 (single-occupancy constraint)
sh_basis = spin_half_basis(L)
print(f'spin_half_basis(L={L}):  {len(sh_basis)} states  (expected 2^{L}={2**L})')
print()
print(f'{"idx":>4}  {"(alpha_bits, beta_bits)":>26}  ket')
print('-'*52)
for i, (a, b) in enumerate(sh_basis):
    ket = '|' + ','.join('↑' if (a>>k)&1 else '↓' for k in range(L)) + '⟩'
    print(f'{i:>4}  {str((a, b)):>26}  {ket}')

spin_half_basis(L=2):  4 states  (expected 2^2=4)

 idx     (alpha_bits, beta_bits)  ket
----------------------------------------------------
   0                      (0, 3)  |↓,↓⟩
   1                      (1, 2)  |↑,↓⟩
   2                      (2, 1)  |↓,↑⟩
   3                      (3, 0)  |↑,↑⟩


---
## 3  Spin operator matrices

Using the single-occupancy spin-½ basis we build the $2^L \times 2^L$ matrix representations of $S_k^+$, $S_k^-$, $S_k^z$ for each site $k$.

The matrix element $M_{ij}$ equals the amplitude with which operator $\hat{O}$ maps basis state $|j\rangle$ to basis state $|i\rangle$:
$$M_{ij} = \langle i | \hat{O} | j \rangle.$$

In [3]:
# Build spin operator matrices on the spin-½ basis (L=2)
L = 2
basis = spin_half_basis(L)

Sp = [spin_plus_matrix(basis, k)  for k in range(L)]
Sm = [spin_minus_matrix(basis, k) for k in range(L)]
Sz = [spin_z_matrix(basis, k)     for k in range(L)]

for k in range(L):
    print(f'── site {k} ─────────────────')
    print(f'S+:\n{Sp[k].real}')
    print(f'S-:\n{Sm[k].real}')
    print(f'Sz:\n{Sz[k]}\n')

── site 0 ─────────────────
S+:
[[ 0.  0.  0.  0.]
 [ 1.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0. -1.  0.]]
S-:
[[ 0.  1.  0.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0. -1.]
 [ 0.  0.  0.  0.]]
Sz:
[[-0.5  0.   0.   0. ]
 [ 0.   0.5  0.   0. ]
 [ 0.   0.  -0.5  0. ]
 [ 0.   0.   0.   0.5]]

── site 1 ─────────────────
S+:
[[ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [-1.  0.  0.  0.]
 [ 0.  1.  0.  0.]]
S-:
[[ 0.  0. -1.  0.]
 [ 0.  0.  0.  1.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]]
Sz:
[[-0.5  0.   0.   0. ]
 [ 0.  -0.5  0.   0. ]
 [ 0.   0.   0.5  0. ]
 [ 0.   0.   0.   0.5]]



**Interpretation (L=2, basis order: |↓↓⟩, |↑↓⟩, |↓↑⟩, |↑↑⟩):**

* $S_0^+$ flips the spin at **site 0** from ↓ → ↑ (maps |↓↓⟩→|↑↓⟩ and |↓↑⟩→|↑↑⟩).
* $S_0^-$ does the reverse.
* $S_0^z$ is diagonal: $+\tfrac{1}{2}$ when site 0 is ↑, $-\tfrac{1}{2}$ when ↓.

These reproduce the Pauli-matrix blocks acting on site 0, embedded in the many-body Hilbert space.

---
## 4  Verification of spin algebra commutation relations

`verify_commutations` checks the three $\mathfrak{su}(2)$ relations numerically using the Frobenius norm of the residual:

$$\|[S^+, S^-] - 2S^z\|_F, \quad \|[S^z, S^+] - S^+\|_F, \quad \|[S^z, S^-] + S^-\|_F.$$

A result below $10^{-10}$ confirms machine-precision agreement.

In [4]:
for L_test in [2, 3, 4]:
    basis_test = spin_half_basis(L_test)
    print(f'L = {L_test}  (basis dim = {len(basis_test)})')
    for site in range(L_test):
        res = verify_commutations(basis_test, site)
        for name, info in res.items():
            ok = '✔ PASS' if info['passed'] else '✘ FAIL'
            print(f'  site {site}  {name:<22}  error = {info["max_error"]:.2e}  {ok}')
    print()

L = 2  (basis dim = 4)
  site 0  [S+, S-] = 2Sz          error = 0.00e+00  ✔ PASS
  site 0  [Sz, S+] = +S+          error = 0.00e+00  ✔ PASS
  site 0  [Sz, S-] = -S-          error = 0.00e+00  ✔ PASS
  site 1  [S+, S-] = 2Sz          error = 0.00e+00  ✔ PASS
  site 1  [Sz, S+] = +S+          error = 0.00e+00  ✔ PASS
  site 1  [Sz, S-] = -S-          error = 0.00e+00  ✔ PASS

L = 3  (basis dim = 8)
  site 0  [S+, S-] = 2Sz          error = 0.00e+00  ✔ PASS
  site 0  [Sz, S+] = +S+          error = 0.00e+00  ✔ PASS
  site 0  [Sz, S-] = -S-          error = 0.00e+00  ✔ PASS
  site 1  [S+, S-] = 2Sz          error = 0.00e+00  ✔ PASS
  site 1  [Sz, S+] = +S+          error = 0.00e+00  ✔ PASS
  site 1  [Sz, S-] = -S-          error = 0.00e+00  ✔ PASS
  site 2  [S+, S-] = 2Sz          error = 0.00e+00  ✔ PASS
  site 2  [Sz, S+] = +S+          error = 0.00e+00  ✔ PASS
  site 2  [Sz, S-] = -S-          error = 0.00e+00  ✔ PASS

L = 4  (basis dim = 16)
  site 0  [S+, S-] = 2Sz          error = 0

All three $\mathfrak{su}(2)$ relations hold to machine precision for every site and every system size tested. ✔

---
## 5  XYZ Heisenberg Hamiltonian: 2–4 site chain

We use `build_xyz_hamiltonian` to build the Hamiltonian matrix.  

> **Coupling convention reminder:** `build_xyz_hamiltonian` sums over all ordered pairs $(p,q)$, so for a symmetric $J$ matrix each physical bond is counted **twice**.  We use $J_{pq} = J_{\text{phys}}/2$ here so that the resulting spectrum matches the standard Heisenberg convention $H = J_{\text{phys}} \sum_{\langle pq\rangle} \mathbf{S}_p \cdot \mathbf{S}_q$.

Below we build open (non-periodic) nearest-neighbour chains for $L = 2, 3, 4$.


In [5]:
def make_chain_J(L, Jx_phys, Jy_phys, Jz_phys):
    """Return symmetric coupling matrices for an open chain.

    Entries are halved (J_phys/2) to compensate for the double-counting
    in build_xyz_hamiltonian, so the spectrum matches the standard
    convention J_phys * Sum_{<pq>} S_p . S_q.
    """
    Jx = np.zeros((L, L))
    Jy = np.zeros((L, L))
    Jz = np.zeros((L, L))
    for p in range(L - 1):
        Jx[p, p+1] = Jx[p+1, p] = Jx_phys / 2
        Jy[p, p+1] = Jy[p+1, p] = Jy_phys / 2
        Jz[p, p+1] = Jz[p+1, p] = Jz_phys / 2
    return Jx, Jy, Jz


J_phys = 1.0   # antiferromagnetic
print(f'Open XXX chain: J_phys = {J_phys}')
print(f'{"L":>4}  {"dim":>6}  {"E0":>14}  {"E0/L":>10}')
print('-' * 40)
for L in [2, 3, 4]:
    basis = spin_half_basis(L)
    Jx, Jy, Jz = make_chain_J(L, J_phys, J_phys, J_phys)
    mu = np.zeros(L)
    H = build_xyz_hamiltonian(L, Jx, Jy, Jz, mu, basis)
    assert np.max(np.abs(H.imag)) < 1e-12, 'H has imaginary parts'
    evals = np.linalg.eigvalsh(H.real)
    print(f'{L:>4}  {len(basis):>6}  {evals[0]:>14.8f}  {evals[0]/L:>10.8f}')

Open XXX chain: J_phys = 1.0
   L     dim              E0        E0/L
----------------------------------------
   2       4     -0.75000000  -0.37500000
   3       8     -1.00000000  -0.33333333
   4      16     -1.61602540  -0.40400635


---
## 6  Benchmark against Pauli-matrix construction

As an independent check we build the same XXX Hamiltonian using the Kronecker-product Pauli-matrix method (identical to `Ising.ipynb`) and compare the full eigenvalue spectra.  For this comparison we use **open chains** of $L = 2, 3, 4$ sites.


In [6]:
# ── Pauli-matrix reference construction ──────────────────────────────────────
sx = np.array([[0,  1 ], [ 1, 0 ]], dtype=complex) / 2
sy = np.array([[0, -1j], [1j, 0 ]], dtype=complex) / 2
sz = np.array([[1,  0 ], [ 0,-1 ]], dtype=complex) / 2
I2 = np.eye(2, dtype=complex)

def site_op(mat, site, L):
    """Embed 2×2 matrix into L-site space by Kronecker product."""
    ops = [I2] * L
    ops[site] = mat
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def build_xxx_pauli(L, J_phys=1.0):
    """Open (non-periodic) XXX chain via Pauli matrices."""
    H = np.zeros((2**L, 2**L), dtype=complex)
    for p in range(L - 1):
        q = p + 1
        H += J_phys * (site_op(sx, p, L) @ site_op(sx, q, L)
                     + site_op(sy, p, L) @ site_op(sy, q, L)
                     + site_op(sz, p, L) @ site_op(sz, q, L))
    return H


# ── Compare spectra ──────────────────────────────────────────────────────────
J_phys = 1.0
print(f'{"L":>3}  {"E0 (Fock)": >16}  {"E0 (Pauli)": >16}  {"max |Δλ|": >12}')
print('-' * 54)
for L in [2, 3, 4]:
    # Fock-space approach
    basis = spin_half_basis(L)
    Jx, Jy, Jz = make_chain_J(L, J_phys, J_phys, J_phys)
    H_fock  = build_xyz_hamiltonian(L, Jx, Jy, Jz, np.zeros(L), basis)
    ev_fock = np.sort(np.linalg.eigvalsh(H_fock.real))

    # Pauli-matrix approach
    H_pauli  = build_xxx_pauli(L, J_phys)
    ev_pauli = np.sort(np.linalg.eigvalsh(H_pauli.real))

    diff = np.max(np.abs(ev_fock - ev_pauli))
    ok = '✔' if diff < 1e-10 else '✘'
    print(f'{L:>3}  {ev_fock[0]:>16.10f}  {ev_pauli[0]:>16.10f}  {diff:>12.2e}  {ok}')

print('\nFock and Pauli constructions produce identical spectra. ✔')

  L         E0 (Fock)        E0 (Pauli)      max |Δλ|
------------------------------------------------------
  2     -0.7500000000     -0.7500000000      0.00e+00  ✔
  3     -1.0000000000     -1.0000000000      0.00e+00  ✔
  4     -1.6160254038     -1.6160254038      0.00e+00  ✔

Fock and Pauli constructions produce identical spectra. ✔


---
## 7  Special limits

### 7.1  Ising limit ($J_x = J_y = 0$)

With transverse couplings disabled, the XYZ Hamiltonian reduces to

$$\hat{H}_{\text{Ising}} = J^z \sum_{\langle pq\rangle} S_p^z S_q^z.$$

Because $S^z$ is diagonal in the occupation-number basis, the Hamiltonian must be diagonal — each basis state is already an eigenstate.

In [7]:
L = 4
basis = spin_half_basis(L)
Jx, Jy, Jz = make_chain_J(L, 0.0, 0.0, 1.0)
H_ising = build_xyz_hamiltonian(L, Jx, Jy, Jz, np.zeros(L), basis)

off_diag = np.max(np.abs(H_ising - np.diag(np.diag(H_ising))))
print(f'Max off-diagonal element: {off_diag:.2e}  [PASS]' if off_diag < 1e-12 else 'FAIL')

evals = np.sort(np.linalg.eigvalsh(H_ising.real))
print(f'Eigenvalue spectrum: {evals}')
print(f'Ground-state energy (J_z = 1, L = 4, open): {evals[0]:.6f}')

Max off-diagonal element: 0.00e+00  [PASS]
Eigenvalue spectrum: [-0.75 -0.75 -0.25 -0.25 -0.25 -0.25 -0.25 -0.25  0.25  0.25  0.25  0.25
  0.25  0.25  0.75  0.75]
Ground-state energy (J_z = 1, L = 4, open): -0.750000


### 7.2  XXX limit ($J_x = J_y = J_z = J$): Bethe-ansatz comparison

For the 1-D antiferromagnetic XXX Heisenberg model in the **thermodynamic limit**, the Bethe ansatz gives:

$$e_\infty = J\left(\tfrac{1}{4} - \ln 2\right) \approx -0.4431\,J.$$

Small open chains converge to this value as $L \to \infty$.

In [8]:
J_phys = 1.0
e_inf = J_phys * (0.25 - np.log(2))   # Bethe ansatz (per site, thermodynamic limit)
print(f'Bethe-ansatz thermodynamic limit: e_inf/J = {e_inf:.8f}')
print()
print(f'{"L":>4}  {"E0/L":>12}  {"error":>12}')
print('-' * 32)
for L in [2, 3, 4]:
    basis = spin_half_basis(L)
    Jx, Jy, Jz = make_chain_J(L, J_phys, J_phys, J_phys)
    H = build_xyz_hamiltonian(L, Jx, Jy, Jz, np.zeros(L), basis)
    evals = np.linalg.eigvalsh(H.real)
    e0_per_site = evals[0] / L
    print(f'{L:>4}  {e0_per_site:>12.8f}  {abs(e0_per_site - e_inf):>12.6f}')
print('\n(Finite-size systems lie above the thermodynamic limit.)')

Bethe-ansatz thermodynamic limit: e_inf/J = -0.44314718

   L          E0/L         error
--------------------------------
   2   -0.37500000      0.068147
   3   -0.33333333      0.109814
   4   -0.40400635      0.039141

(Finite-size systems lie above the thermodynamic limit.)


### 7.3  Anisotropic XYZ: effect of $J_y \neq J_x$

When $J_x \neq J_y$ the pairing terms $S_p^+ S_q^+ + S_p^- S_q^-$ appear, which change the total $S^z$ by $\pm 2$.  This is the unique feature of the full XYZ model — **not** accessible through the existing `HamHeisenberg` class.

In [9]:
L = 4
basis = spin_half_basis(L)
mu = np.zeros(L)

print(f'L={L}, fixed J_z=1, J_y=0 (open chain), varying J_x')
print(f'{"Jx_phys":>10}  {"E0":>14}  {"E0/L":>10}')
print('-' * 40)
for jx in [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]:
    Jx, Jy, Jz = make_chain_J(L, jx, 0.0, 1.0)
    H = build_xyz_hamiltonian(L, Jx, Jy, Jz, mu, basis)
    evals = np.linalg.eigvalsh(H.real)
    print(f'{jx:>10.3f}  {evals[0]:>14.8f}  {evals[0]/L:>10.8f}')

L=4, fixed J_z=1, J_y=0 (open chain), varying J_x
   Jx_phys              E0        E0/L
----------------------------------------
     0.000     -0.75000000  -0.18750000
     0.250     -0.78339961  -0.19584990
     0.500     -0.86894159  -0.21723540
     0.750     -0.98469414  -0.24617354
     1.000     -1.11803399  -0.27950850
     1.500     -1.41556942  -0.35389235


---
## 8  Limitations and future work

| Limitation | Current status |
|---|---|
| Fock-space dimension grows as $\binom{L}{N_\alpha} \times \binom{L}{N_\beta}$; practical only for $L \lesssim 5$. | Prototype intentionally kept small. |
| Dense matrices only (no sparse representation). | Planned for GSoC production code. |
| Not yet integrated into `HamiltonianAPI` (`generate_one_body_integral` / `generate_two_body_integral`). | **Primary GSoC deliverable.** |
| No translational or $S^z$-symmetry block-diagonalisation. | Planned. |

The core algebraic correctness (commutation relations, spectral benchmarks, Bethe-ansatz limit) has been verified above.  Full `HamiltonianAPI` integration will make this available as an official part of the ModelHamiltonian package and extend support to general XYZ Heisenberg models.
